# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muhammadfarhan2157-source/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*
Finding A — "What Predicts Health?" (Random Forest feature importance, page 27)

The paper reports Average Position (43%), Impressions (32%), and Scroll Depth (15%) as the top predictors of health score, and to its credit, flags this itself: "health score already includes some visibility inputs... high importance is therefore expected and does not imply external causation."

My methodology question: where does the label come from, relative to the features? Health score is explicitly defined as Impressions (30pts) + Position (30pts) + CTR (20pts) + Scroll Depth (20pts) — and the model's top three predictors (position, impressions, scroll) are three of the four components that literally compose the label. This matches leakage taxonomy type 1 from the leakage skill almost exactly: "the label was computed FROM a column, and that column is in the features... one feature towers over all others." The paper's own caveat is doing real work here, and I think it's the right caveat — but I'd go one step further and ask: is there a version of this analysis that predicts a genuinely external outcome (future traffic, future health-score change) instead of the current health score itself? That would let the feature importance ranking mean something closer to "predicts what drives improvement" rather than "recovers the known formula."

Finding B — "What Predicts Growth?" (Logistic Regression, 71% holdout accuracy, page 29)

The paper reports 71% holdout accuracy for the growth/decline classifier, without stating the class balance next to it.

My methodology question: what's the base rate, and does the validation design support the claim at face value? From Finding #1's own numbers (74.8K growing vs. 45.6K declining), the majority-class base rate is roughly 62%. That means 71% accuracy is a real but modest 9-point lift over guessing the majority class every time — not the strong result "71% accuracy" sounds like in isolation. This is almost the textbook example from the leakage-and-validation skill itself ("accuracy of 71% on a label that's 62% positive is 9 points still, not 71"). I'd ask: could the paper report accuracy next to the base rate everywhere a classifier result appears, so readers can judge the real lift rather than the raw number? To be fair to the paper, it does hedge appropriately elsewhere ("read these as descriptive indicators... not direct instructions"), so this is a presentation gap more than an overclaim — the kind of thing worth flagging exactly because the paper is otherwise careful.

In [1]:

from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")

con.sql(f"""
    CREATE SECRET hf_token (
        TYPE HUGGINGFACE,
        TOKEN '{HF_TOKEN}'
    );
""")

path_march = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"
path_april = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet"

# quick test — this should actually hit the dataset
test = con.sql(f"SELECT * FROM '{path_march}' LIMIT 3").df()
print("Connected and read succeeded.")
test

Connected and read succeeded.


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

Before: retraining the same Week-5 features/model but with a random (non-grouped) split — the same client's pages can appear in both train and test. After: the actual grouped, client-holdout split from Week-5. Comparing Precision@50 and ROC-AUC on both to show what the ungrouped split was silently inflating.

In [3]:
march = con.sql(f"""
    SELECT content_hash_id, client_hash_id,
           SUM(gsc_impressions) AS impressions_march,
           SUM(gsc_clicks) AS clicks_march,
           AVG(gsc_avg_position) AS avg_position_march,
           SUM(gsc_clicks)/NULLIF(SUM(gsc_impressions),0) AS ctr_march
    FROM '{path_march}'
    WHERE gsc_data_available IS TRUE
    GROUP BY 1,2
""").df()

april = con.sql(f"""
    SELECT content_hash_id, client_hash_id, SUM(gsc_clicks) AS clicks_april
    FROM '{path_april}'
    WHERE gsc_data_available IS TRUE
    GROUP BY 1,2
""").df()

data = march.merge(april, on=["content_hash_id", "client_hash_id"])
data["declined"] = (data["clicks_april"] < data["clicks_march"]).astype(int)

print("data rebuilt:", data.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

data rebuilt: (158549, 8)


In [4]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
import pandas as pd

feature_cols = ["impressions_march", "avg_position_march", "ctr_march"]

# BEFORE: naive random split, no client grouping
train_naive, test_naive = train_test_split(data, test_size=0.3, random_state=1)
rf_naive = RandomForestClassifier(n_estimators=200, random_state=1).fit(train_naive[feature_cols], train_naive["declined"])
naive_scores = rf_naive.predict_proba(test_naive[feature_cols])[:, 1]
naive_overlap = set(train_naive["client_hash_id"]) & set(test_naive["client_hash_id"])

# AFTER: grouped split (same as w05)
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=1)
train_idx, test_idx = next(gss.split(data, groups=data["client_hash_id"]))
train_grouped, test_grouped = data.iloc[train_idx], data.iloc[test_idx]
rf_grouped = RandomForestClassifier(n_estimators=200, random_state=1).fit(train_grouped[feature_cols], train_grouped["declined"])
grouped_scores = rf_grouped.predict_proba(test_grouped[feature_cols])[:, 1]
grouped_overlap = set(train_grouped["client_hash_id"]) & set(test_grouped["client_hash_id"])

def precision_at_k(y_true, scores, k=50):
    top_k_idx = pd.Series(scores).nlargest(k).index
    return y_true.iloc[top_k_idx].mean()

comparison = pd.DataFrame({
    "split": ["Naive random (before)", "Grouped by client (after)"],
    "client_overlap_count": [len(naive_overlap), len(grouped_overlap)],
    "roc_auc": [
        roc_auc_score(test_naive["declined"], naive_scores),
        roc_auc_score(test_grouped["declined"], grouped_scores),
    ],
    "precision_at_50": [
        precision_at_k(test_naive["declined"].reset_index(drop=True), pd.Series(naive_scores)),
        precision_at_k(test_grouped["declined"].reset_index(drop=True), pd.Series(grouped_scores)),
    ],
})
comparison

,split,client_overlap_count,roc_auc,precision_at_50
0,Naive random (before),44,0.912880,0.98
1,Grouped by client (after),0,0.888801,0.90


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [8]:
print("Final feature columns used in the model:", feature_cols)

leakage_checklist = {
    "Any feature calculated after the decision point?": "No — all features are averaged from March 2026 only; label comes from April 2026.",
    "Does the feature window overlap the target window?": "No — March (features) and April (label) are disjoint months.",
    "Any product decision flag used as a feature?": "No — health_score, priority_score, action_type, needs_ctr_fix, is_quick_win are all absent from feature_cols.",
    "Does any derived field secretly encode the target?": "[Check: does ctr_march or avg_position_march correlate suspiciously perfectly with 'declined'? Run a quick correlation check below.]",
    "Are duplicate/related rows split across train/test?": "Checked in Section 2 — grouped split confirms 0 client overlap.",
}
for q, a in leakage_checklist.items():
    print(f"- {q}\n  {a}\n")

print(data[feature_cols + ["declined"]].corr()["declined"].sort_values(ascending=False))

print("\n" + "="*60)
print("THE ATTACK: deliberately add a leaky feature, watch it collapse toward 1.0")
print("="*60 + "\n")

data_leaky = data.copy()
data_leaky["clicks_april_leaked"] = data_leaky["clicks_april"]

leaky_features = feature_cols + ["clicks_april_leaked"]
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=1)
tr_idx, te_idx = next(gss2.split(data_leaky, groups=data_leaky["client_hash_id"]))
train_l, test_l = data_leaky.iloc[tr_idx], data_leaky.iloc[te_idx]

rf_leaky = RandomForestClassifier(n_estimators=200, random_state=1).fit(train_l[leaky_features], train_l["declined"])
leaky_scores = rf_leaky.predict_proba(test_l[leaky_features])[:, 1]

print("AUC WITH leak:", roc_auc_score(test_l["declined"], leaky_scores))
print("AUC WITHOUT leak (honest, from Section 2):", roc_auc_score(test_grouped["declined"], grouped_scores))
print("Collapse from ~1.0 back to the honest number IS the confession, per the skill.")

Final feature columns used in the model: ['impressions_march', 'avg_position_march', 'ctr_march']
- Any feature calculated after the decision point?
  No — all features are averaged from March 2026 only; label comes from April 2026.

- Does the feature window overlap the target window?
  No — March (features) and April (label) are disjoint months.

- Any product decision flag used as a feature?
  No — health_score, priority_score, action_type, needs_ctr_fix, is_quick_win are all absent from feature_cols.

- Does any derived field secretly encode the target?
  [Check: does ctr_march or avg_position_march correlate suspiciously perfectly with 'declined'? Run a quick correlation check below.]

- Are duplicate/related rows split across train/test?
  Checked in Section 2 — grouped split confirms 0 client overlap.

declined              1.000000
impressions_march     0.219581
ctr_march             0.163866
avg_position_march   -0.184235
Name: declined, dtype: float64

THE ATTACK: deliberatel

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

The paper itself is a good model for this. Look at how it hedges Finding #10 (AI model comparison): "This section is best read as a process comparison, not a victory lap for one provider family" — and Finding #8: "should not be used as a headline decay proof point because the active-content subset introduces strong survivor bias there." That's the register I'm matching: name the result, then name exactly what it doesn't prove, in the same sentence or the next one.

My boldest claim from Week 5: "The random forest beats the baseline rule."

Before: "The model beats the baseline."
After (matching the paper's own hedging pattern): "On this client-holdout split, the random forest was observed to reach a higher Precision@50 than the Week-4 baseline rule — this is decision-support evidence for this slice and time window, not a claim that holds automatically on other months or at full warehouse scale, the same way the paper treats its 361+ freshness bucket as directionally present but too small to be a headline multiplier."

In [9]:
paper_hedge_examples = [
    "This section is best read as a process comparison, not a victory lap for one provider family.",
    "should not be used as a headline decay proof point because the active-content subset introduces strong survivor bias",
    "Read these as descriptive indicators from the sampled active-content set, not as direct instructions to optimize one variable in isolation.",
]

claim_before = "The random forest model beats the hand-written baseline rule."
claim_after = (
    "On a client-holdout split using March 2026 features and April 2026 outcomes, the random forest "
    "was observed to reach a higher Precision@50 than the Week-4 baseline rule. This is directional, "
    "decision-support evidence for this specific slice and time window — not a claim that holds "
    "automatically on other months, other clients, or at full warehouse scale."
)

print("Paper's hedging style (for reference):")
for ex in paper_hedge_examples:
    print(" -", ex)
print("\nBEFORE:", claim_before)
print("AFTER:", claim_after)


Paper's hedging style (for reference):
 - This section is best read as a process comparison, not a victory lap for one provider family.
 - should not be used as a headline decay proof point because the active-content subset introduces strong survivor bias
 - Read these as descriptive indicators from the sampled active-content set, not as direct instructions to optimize one variable in isolation.

BEFORE: The random forest model beats the hand-written baseline rule.
AFTER: On a client-holdout split using March 2026 features and April 2026 outcomes, the random forest was observed to reach a higher Precision@50 than the Week-4 baseline rule. This is directional, decision-support evidence for this specific slice and time window — not a claim that holds automatically on other months, other clients, or at full warehouse scale.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.